# FusionMatch — Phase 2: Model Development & Sanity Checks

This notebook demonstrates and validates the **FusionMatch** cross-modal model architecture:
1. **SigLIP Dual Encoder Backbone**: Vision & Text towers with parameter freeze/unfreeze controls.
2. **Multi-View Visual Aggregation**: Mean-pooling across $K$ angle perspectives per SKU.
3. **Quality-Aware Gated Fusion**: Dynamic softmax gating conditioned on multimodal features and Laplacian/length quality proxies ($g_v + g_t = 1$).
4. **Contrastive Projection Head**: MLP projecting fused representations to unit-norm 256-d hypersphere vectors.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import (
    FusionMatchModel,
    SiglipDualEncoder,
    GatedFusion,
    ProjectionHead,
    image_quality_score,
    text_quality_score,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Model Instantiation & Parameter Budget Inspection

In Phase 1 of training (warm-up), the SigLIP backbone is frozen, training only the gated fusion module and projection head (~1.7M trainable parameters). In Phase 2 (fine-tuning), the last 2 transformer blocks of each tower are unfrozen (~31.7M trainable parameters).

In [ ]:
# Model with frozen backbone (Phase 1 warm-up configuration)
model_warmup = FusionMatchModel(use_mock=True, freeze_vision=True, freeze_text=True, embed_dim=256)
budget_warmup = model_warmup.get_param_budget_summary()

# Model with unfrozen last 2 blocks (Phase 2 fine-tuning configuration)
model_finetune = FusionMatchModel(use_mock=True, freeze_vision=False, freeze_text=False, unfreeze_last_n_blocks=2, embed_dim=256)
budget_finetune = model_finetune.get_param_budget_summary()

print("=== WARM-UP PARAMETER BUDGET ===")
print(f"Total Parameters:     {budget_warmup['total_params']:,}")
print(f"Trainable Parameters: {budget_warmup['trainable_params']:,} ({budget_warmup['trainable_params']/budget_warmup['total_params']*100:.2f}%)")
print(f"  - Gated Fusion:     {budget_warmup['fusion']['trainable']:,} trainable / {budget_warmup['fusion']['total']:,} total")
print(f"  - Projection Head:  {budget_warmup['projection_head']['trainable']:,} trainable / {budget_warmup['projection_head']['total']:,} total")
print(f"  - Encoder Backbone: {budget_warmup['encoder']['trainable']:,} trainable / {budget_warmup['encoder']['total']:,} total")

print("\n=== FINE-TUNING PARAMETER BUDGET ===")
print(f"Total Parameters:     {budget_finetune['total_params']:,}")
print(f"Trainable Parameters: {budget_finetune['trainable_params']:,} ({budget_finetune['trainable_params']/budget_finetune['total_params']*100:.2f}%)")

## 2. Quality Proxy Estimators

We evaluate the sensitivity of heuristic quality proxies across sharp vs. blurred images and dense vs. sparse/empty text.

In [ ]:
# Synthesize image quality test cases
sharp_img = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
smooth_img = np.ones((256, 256, 3), dtype=np.uint8) * 128

q_sharp = image_quality_score([sharp_img]).item()
q_smooth = image_quality_score([smooth_img]).item()

# Synthesize text quality test cases
text_samples = [
    "AmazonBasics 22-Inch Bottom Mount Drawer Slides, White Powder Coat Steel, 10-Pair Pack",
    "Drawer Slides",
    "",
]
q_texts = text_quality_score(text_samples).tolist()

print(f"Sharp Image Quality Proxy:  {q_sharp:.4f}")
print(f"Smooth/Blur Quality Proxy:  {q_smooth:.4f}")
print("\nText Quality Proxies:")
for t, q in zip(text_samples, q_texts):
    print(f"  - Quality: {q:.4f} | Text: '{t}'")

## 3. Quality-Aware Gated Fusion Behavior

Testing how the gating network adjusts modality contributions ($g_v, g_t$) when visual quality is degraded versus when text quality is degraded.

In [ ]:
fusion = GatedFusion(vision_dim=768, text_dim=768, shared_dim=768)

b = 3
v_dummy = torch.randn(b, 768)
t_dummy = torch.randn(b, 768)

# Scenario 1: High vision quality, low text quality
# Scenario 2: Low vision quality, high text quality
# Scenario 3: Balanced quality
q_v = torch.tensor([0.95, 0.10, 0.80])
q_t = torch.tensor([0.10, 0.95, 0.80])

fused, gates = fusion(v_dummy, t_dummy, q_v, q_t)

print("=== GATING NETWORK OUTPUTS ===")
for i in range(b):
    g_v = gates[i, 0].item()
    g_t = gates[i, 1].item()
    print(f"Sample {i+1}: q_v={q_v[i]:.2f}, q_t={q_t[i]:.2f} -> Vision Gate g_v={g_v:.4f}, Text Gate g_t={g_t:.4f} (Sum: {g_v+g_t:.4f})")

## 4. Multi-View Visual Perspective Aggregation

Verifying that multi-angle photography $(B, K, 3, H, W)$ per SKU is mean-pooled into a single representative vector $(B, 256)$ with unit length on the hypersphere.

In [ ]:
b, k = 4, 3  # 4 SKUs, 3 multi-angle images per SKU
pixel_values_multiview = torch.randn(b, k, 3, 224, 224)
input_ids = torch.randint(0, 1000, (b, 24))
attention_mask = torch.ones(b, 24, dtype=torch.long)
q_v = torch.rand(b)
q_t = torch.rand(b)

emb, gates = model_warmup(pixel_values_multiview, input_ids, attention_mask, q_v, q_t)

print(f"Multi-view input shape:  {pixel_values_multiview.shape}")
print(f"Output embedding shape:  {emb.shape}")
print(f"Output gates shape:      {gates.shape}")
print(f"Embedding L2 Norms:      {torch.norm(emb, p=2, dim=-1).tolist()}")
assert torch.allclose(torch.norm(emb, p=2, dim=-1), torch.ones(b), atol=1e-4), "Embeddings must have unit L2 norm!"

## 5. End-to-End Real Batch Forward Pass

We load a real batch from `FusionMatchDataset` using our manifest files generated in Phase 1.

In [ ]:
from src.data.dataset import FusionMatchDataset, collate_fusion_match_batch
from torch.utils.data import DataLoader

manifest_path = project_root / "data" / "processed" / "manifest_train.csv"
if manifest_path.exists():
    dataset = FusionMatchDataset(manifest_path=manifest_path, is_training=False)
    loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_fusion_match_batch)
    
    batch = next(iter(loader))
    print(f"Batch pixel_values shape: {batch['pixel_values'].shape}")
    print(f"Batch input_ids shape:    {batch['input_ids'].shape}")
    print(f"Batch q_v values:         {batch['q_v'].tolist()}")
    print(f"Batch q_t values:         {batch['q_t'].tolist()}")
    
    emb, gates = model_warmup(
        pixel_values=batch["pixel_values"],
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        q_v=batch["q_v"],
        q_t=batch["q_t"],
    )
    print(f"\nGenerated Embeddings shape: {emb.shape}")
    print(f"Generated Gate weights:\n{gates}")
    print("\nPhase 2 Model Sanity Checks Successfully Completed!")
else:
    print("Manifest file not found, please run Phase 1 first.")